# Data Collection for Reinforcement Learning in Finance

This notebook collects and prepares financial data for reinforcement learning applications. It downloads historical price data for a portfolio of stocks, filters out young assets, and creates technical features for machine learning.

## Overview
- Load portfolio holdings from CSV
- Download 20-year historical data using yfinance
- Filter assets based on data availability (minimum 5 years)
- Calculate technical indicators and features
- Save processed data for training

## Key Features Calculated
- Log returns
- Simple Moving Averages (10-day and 30-day)
- Volatility measures
- Sharpe ratios (60-day and 120-day windows)
- Rolling statistics for returns

## 1. Import Required Libraries

In [18]:
import pandas as pd
import yfinance as yf
import numpy as np
import warnings
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

datetime.now().strftime('%Y-%m-%d %H:%M:%S')

'2025-08-07 09:35:02'

## 2. Load Portfolio and Download Price Data

In [19]:
# Load portfolio holdings from CSV file

portfolio = pd.read_csv("portfolio_holdings.csv")
tickers = portfolio['Ticker'].unique().tolist()
print(f" Portfolio loaded successfully with {len(tickers)} unique tickers")
print(f"Tickers: {tickers}")


# Download historical price data
PERIOD = '20y' # 20 years of data
print(f"\n Downloading {PERIOD} of historical data for {len(tickers)} tickers...")

data = yf.download(tickers, period=PERIOD, auto_adjust=True)['Close'].round(2)
print(f" Data download completed. Shape: {data.shape}")

[***********           22%                       ]  4 of 18 completed

 Portfolio loaded successfully with 18 unique tickers
Tickers: ['RDDT', 'NVDA', 'SMR', 'MU', 'MRVL', 'MSFT', 'ASML', 'AEM', 'AMD', 'VERU', 'AI', 'GOOGL', 'INGM', 'PLUG', 'IONQ', 'CHYM', 'RGTI', 'ARBE']



[*********************100%***********************]  18 of 18 completed

 Data download completed. Shape: (5031, 18)


In [20]:
data.head()

Ticker,AEM,AI,AMD,ARBE,ASML,CHYM,GOOGL,INGM,IONQ,MRVL,MSFT,MU,NVDA,PLUG,RDDT,RGTI,SMR,VERU
Date,,,,,,,,,,,,,,,,,,
2005-08-08,9.91,NaN,19.96,NaN,16.57,NaN,7.25,NaN,NaN,18.23,18.87,11.29,0.22,63.4,NaN,NaN,NaN,1.28
2005-08-09,10.02,NaN,20.42,NaN,16.77,NaN,7.25,NaN,NaN,18.86,19.03,11.27,0.22,64.5,NaN,NaN,NaN,1.24
2005-08-10,10.05,NaN,20.60,NaN,16.73,NaN,7.11,NaN,NaN,18.47,18.75,10.99,0.21,62.8,NaN,NaN,NaN,1.34
2005-08-11,10.39,NaN,20.69,NaN,16.87,NaN,7.07,NaN,NaN,18.80,18.97,11.04,0.22,62.8,NaN,NaN,NaN,1.29
2005-08-12,10.38,NaN,20.70,NaN,16.65,NaN,7.21,NaN,NaN,19.05,18.82,10.85,0.23,63.1,NaN,NaN,NaN,1.26


In [21]:

print(f" RL Data Summary:")
print(f"Shape: {data.shape}")
print(f"Date range: {data.index.min()} to {data.index.max()}")
print(f"Number of trading days: {len(data)}")
print(f"\nLatest prices:")
data.tail()

 RL Data Summary:
Shape: (5031, 18)
Date range: 2005-08-08 00:00:00 to 2025-08-06 00:00:00
Number of trading days: 5031

Latest prices:


Ticker,AEM,AI,AMD,ARBE,ASML,CHYM,GOOGL,INGM,IONQ,MRVL,MSFT,MU,NVDA,PLUG,RDDT,RGTI,SMR,VERU
Date,,,,,,,,,,,,,,,,,,
2025-07-31,124.36,23.56,176.31,1.35,694.71,34.41,191.90,19.72,39.87,80.37,533.50,109.14,177.87,1.50,160.59,14.50,50.21,0.48
2025-08-01,125.22,22.43,171.70,1.32,689.82,32.63,189.13,19.43,38.12,74.45,524.11,104.88,173.72,1.40,188.64,14.12,44.43,0.49
2025-08-04,131.39,23.19,176.78,1.37,699.36,33.88,195.04,19.52,39.86,76.53,535.64,107.77,180.00,1.45,201.76,15.76,43.38,0.48
2025-08-05,134.55,23.29,174.31,1.38,689.63,33.80,194.67,19.23,42.02,76.63,527.75,109.06,178.26,1.54,199.09,16.47,45.13,0.45
2025-08-06,135.42,23.25,163.12,1.37,690.96,33.49,196.09,18.88,41.23,75.32,524.94,108.78,179.42,1.46,212.81,15.99,45.93,0.37


## 5. Calculate Technical Features

Calculate various technical indicators that will be used as features for the reinforcement learning model:

- **Log Returns**: Natural logarithm of price ratios
- **Simple Moving Averages**: 10-day and 30-day SMAs
- **Volatility**: 10-day rolling standard deviation of returns
- **Sharpe Ratios**: Risk-adjusted return measures (60-day and 120-day windows)
- **Rolling Statistics**: Mean and standard deviation of returns over different periods

In [22]:
# Calculate log returns (natural logarithm of price ratios)
log_returns = np.log(data / data.shift(1)).fillna(0)
# Calculate Simple Moving Averages (SMA) relative to current price
# These are normalized by current price to make them comparable across assets
sma10 = data.rolling(10).mean() / data - 1 # 10-day SMA
sma30 = data.rolling(30).mean() / data - 1 # 30-day SMA

# Calculate volatility (10-day rolling standard deviation of returns)
volatility_10 = log_returns.rolling(10).std().fillna(0)

# Calculate 60-day rolling statistics
log_return_mean_60 = log_returns.rolling(60).mean()
log_return_std_60 = log_returns.rolling(60).std()
# Sharpe ratio = mean return / standard deviation (with small epsilon to avoid division by zero)
sharpe_60 = log_return_mean_60 / (log_return_std_60 + 1e-8)

# Calculate 120-day rolling statistics
log_return_mean_120 = log_returns.rolling(120).mean()
log_return_std_120 = log_returns.rolling(120).std()
sharpe_120 = log_return_mean_120 / (log_return_std_120 + 1e-8)

print(" Technical features calculated successfully")
print(f"Features calculated for {len(tickers)} assets over {len(data)} trading days")

 Technical features calculated successfully
Features calculated for 18 assets over 5031 trading days


## 6. Create Feature Dataset for Machine Learning

Transform the data into a long-form dataset suitable for machine learning and reinforcement learning applications. Each row represents a single asset on a single date with all calculated features.

In [23]:
# Stack features into a single DataFrame in long form for ML/RL
feature_frames = []

for ticker in data:
    df = pd.DataFrame({
        'Date': log_returns.index,
        'Ticker': ticker,
        'log_return': log_returns[ticker].values,
        'sma10': sma10[ticker].values,
        'sma30': sma30[ticker].values,
        'volatility_10': volatility_10[ticker].values,
        'log_return_mean_60': log_return_mean_60[ticker].values,
        'log_return_std_60': log_return_std_60[ticker].values,
        'sharpe_60': sharpe_60[ticker].values,
        'log_return_mean_120': log_return_mean_120[ticker].values,
        'log_return_std_120': log_return_std_120[ticker].values,
        'sharpe_120': sharpe_120[ticker].values
    })
    feature_frames.append(df.round(4))

# Combine all feature frames
features = pd.concat(feature_frames, ignore_index=True)

features.fillna(0, inplace=True)

print(f"\n Dataset Summary:")
print(f"Features shape: {features.shape}")
print(f"\n Date range: {features.Date.min()} to {features.Date.max()}")
print(f"\n Sample of features:")
features.head()


 Dataset Summary:
Features shape: (90558, 12)

 Date range: 2005-08-08 00:00:00 to 2025-08-06 00:00:00

 Sample of features:


,Date,Ticker,log_return,sma10,sma30,volatility_10,log_return_mean_60,log_return_std_60,sharpe_60,log_return_mean_120,log_return_std_120,sharpe_120
0,2005-08-08,AEM,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2005-08-09,AEM,0.0110,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2005-08-10,AEM,0.0030,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2005-08-11,AEM,0.0333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2005-08-12,AEM,-0.0010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
prices = data.copy()
prices.fillna(0, inplace=True)

print(f"Price data shape: {prices.shape}")
print(f"\n Date range: {prices.index.min()} to {prices.index.max()}")

# Display sample of price data
print(f"\n Sample of price data:")
prices.head()

Price data shape: (5031, 18)

 Date range: 2005-08-08 00:00:00 to 2025-08-06 00:00:00

 Sample of price data:


Ticker,AEM,AI,AMD,ARBE,ASML,CHYM,GOOGL,INGM,IONQ,MRVL,MSFT,MU,NVDA,PLUG,RDDT,RGTI,SMR,VERU
Date,,,,,,,,,,,,,,,,,,
2005-08-08,9.91,0.0,19.96,0.0,16.57,0.0,7.25,0.0,0.0,18.23,18.87,11.29,0.22,63.4,0.0,0.0,0.0,1.28
2005-08-09,10.02,0.0,20.42,0.0,16.77,0.0,7.25,0.0,0.0,18.86,19.03,11.27,0.22,64.5,0.0,0.0,0.0,1.24
2005-08-10,10.05,0.0,20.60,0.0,16.73,0.0,7.11,0.0,0.0,18.47,18.75,10.99,0.21,62.8,0.0,0.0,0.0,1.34
2005-08-11,10.39,0.0,20.69,0.0,16.87,0.0,7.07,0.0,0.0,18.80,18.97,11.04,0.22,62.8,0.0,0.0,0.0,1.29
2005-08-12,10.38,0.0,20.70,0.0,16.65,0.0,7.21,0.0,0.0,19.05,18.82,10.85,0.23,63.1,0.0,0.0,0.0,1.26


## 7. Save Data for Training

Save the processed data to CSV files for use in subsequent notebooks and training steps.

In [25]:
# Save features and prices for training
features.to_csv("outputs/features_for_training.csv", index=False)
print(f"✓ Features saved: {features.shape}")
prices.to_csv("outputs/price_data_for_training.csv", index=True)
print(f"✓ Prices saved: {prices.shape}")
# Save log returns data for sentiment analysis validation
log_returns.to_csv("outputs/log_returns_data.csv", index=True)
print(f"✓ Log returns saved: {log_returns.shape}")

# Save list of tickers used for RL
with open('rl_tickers.txt', 'w') as f:
    for ticker in tickers:
        f.write(f"{ticker}\n")


print(f"\n Data collection completed successfully at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Features saved: (90558, 12)
✓ Prices saved: (5031, 18)
✓ Log returns saved: (5031, 18)

 Data collection completed successfully at 2025-08-07 09:35:03


## Summary

This notebook successfully:

1. **Loaded portfolio data** from `portfolio_holdings.csv`
2. **Downloaded 20 years** of historical price data for all tickers
3. **Filtered assets** based on data availability (minimum 5 years)
4. **Calculated technical features** including returns, moving averages, volatility, and Sharpe ratios
5. **Created training datasets** in both long-form (features) and wide-form (prices)
6. **Saved processed data** for use in subsequent analysis and training

### Key Statistics:
- **Assets processed**: {len(rl_tickers)} tickers with sufficient historical data
- **Time period**: 20 years of data
- **Features created**: 10 technical indicators per asset
- **Total observations**: {features.shape[0]:,} feature rows

### Next Steps:
The processed data is now ready for:
- Feature engineering and selection
- Model training and validation
- Reinforcement learning algorithm development
- Portfolio optimization strategies